# Notebook 09 — Selective Weighted Ensemble

## What is a Selective Weighted Ensemble?
A selective ensemble combines two models in a way that leverages the strengths of each:
- When both models **agree** on the predicted class → use the primary model (`localization`) directly
- When both models **disagree** → combine predictions via **weighted average**

This approach is grounded in the literature on selective prediction: in clinical settings, when models disagree, the case is inherently more ambiguous and combining predictions reduces the risk of a single model's error propagating to the final decision.

## Why weighted average (and not simple average)?
Weighted average assigns different importance to each model based on their clinical strengths:
- `localization` → weight 0.6 (primary, best Recall Melanoma: 0.7976)
- `sex_age` → weight 0.4 (secondary, best Recall BCC: 0.8442)

Simple average (0.5/0.5) ignores the known performance differences between models. Weighted average is the standard approach in medical imaging ensemble literature (Gessert et al., 2020; Buyrukoğlu et al., 2024).

## Why selective (and not always ensemble)?
When both models agree, combining them adds no information and may dilute the primary model's confidence. Selective ensembling only activates the combination when there is genuine disagreement, preserving the primary model's strengths on clear cases (Lu et al., 2021).

## Models
| Model | Metadata | Role | Key Strength |
|-------|----------|------|-------------|
| `multimodal_b0_none_localization` | localization | Primary | Recall Melanoma: 0.7976 |
| `multimodal_b0_none_sex_age` | sex + age | Secondary | Recall BCC: 0.8442 |

## 4 Experiments evaluated here
| # | Probabilities | Threshold | Notebook |
|---|--------------|-----------|---------|
| 1 | Standard | 0.5 (default) | This notebook |
| 2 | TTA | 0.5 (default) | This notebook |
| 3 | Standard | Adjusted | Notebook 10 |
| 4 | TTA | Adjusted | Notebook 10 |

## Inputs
| File | Location | Description |
|------|----------|-------------|
| `std_probs_localization.npy` | `outputs/metrics/` | Standard probabilities — localization |
| `std_probs_sex_age.npy` | `outputs/metrics/` | Standard probabilities — sex_age |
| `tta_probs_localization.npy` | `outputs/metrics/` | TTA probabilities — localization |
| `tta_probs_sex_age.npy` | `outputs/metrics/` | TTA probabilities — sex_age |
| `std_labels.npy` | `outputs/metrics/` | Ground truth labels |

## Outputs
| File | Location | Description |
|------|----------|-------------|
| `ensemble_std_probs.npy` | `outputs/metrics/` | Ensemble probabilities (standard) |
| `ensemble_tta_probs.npy` | `outputs/metrics/` | Ensemble probabilities (TTA) |
| `ensemble_comparison.png` | `outputs/figures/` | Metrics comparison plot |

In [1]:
import sys
sys.path.append('../src')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, roc_auc_score
)

from config import (
    METRICS_DIR, FIGURES_DIR,
    NUM_CLASSES, IDX_TO_CLASS, CLASSES, CLASS_NAMES_FULL
)

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 120

print('Imports OK')

Imports OK


## Loading Probabilities

In [2]:
# Load probabilities
std_probs_loc = np.load(os.path.join(METRICS_DIR, 'std_probs_localization.npy'))
std_probs_sex = np.load(os.path.join(METRICS_DIR, 'std_probs_sex_age.npy'))
tta_probs_loc = np.load(os.path.join(METRICS_DIR, 'tta_probs_localization.npy'))
tta_probs_sex = np.load(os.path.join(METRICS_DIR, 'tta_probs_sex_age.npy'))
labels        = np.load(os.path.join(METRICS_DIR, 'std_labels.npy'))

print(f'Test samples:         {len(labels)}')
print(f'std_probs_loc shape:  {std_probs_loc.shape}')
print(f'std_probs_sex shape:  {std_probs_sex.shape}')
print(f'tta_probs_loc shape:  {tta_probs_loc.shape}')
print(f'tta_probs_sex shape:  {tta_probs_sex.shape}')

Test samples:         1497
std_probs_loc shape:  (1497, 7)
std_probs_sex shape:  (1497, 7)
tta_probs_loc shape:  (1497, 7)
tta_probs_sex shape:  (1497, 7)


## Selective Weighted Ensemble

In [3]:
# Ensemble weights
W_PRIMARY   = 0.6  # localization
W_SECONDARY = 0.4  # sex_age

def selective_weighted_ensemble(probs_primary, probs_secondary, w1=W_PRIMARY, w2=W_SECONDARY):
    """
    Selective weighted ensemble:
    - If both models agree on predicted class -> use primary model directly
    - If they disagree -> weighted average of probabilities

    Args:
        probs_primary:   Probabilities from primary model (N, NUM_CLASSES)
        probs_secondary: Probabilities from secondary model (N, NUM_CLASSES)
        w1:              Weight for primary model (default 0.6)
        w2:              Weight for secondary model (default 0.4)

    Returns:
        final_probs:     Final ensemble probabilities (N, NUM_CLASSES)
        agree_mask:      Boolean mask — True where models agreed (N,)
    """
    preds_primary   = np.argmax(probs_primary, axis=1)
    preds_secondary = np.argmax(probs_secondary, axis=1)

    agree_mask = preds_primary == preds_secondary

    # Start with weighted average for all samples
    final_probs = w1 * probs_primary + w2 * probs_secondary

    # Override with primary probs where models agree
    final_probs[agree_mask] = probs_primary[agree_mask]

    return final_probs, agree_mask


def compute_metrics(labels, probs):
    """Compute key metrics from probabilities."""
    preds = np.argmax(probs, axis=1)
    return {
        'accuracy':     accuracy_score(labels, preds),
        'recall_macro': recall_score(labels, preds, average='macro', zero_division=0),
        'f1_macro':     f1_score(labels, preds, average='macro', zero_division=0),
        'auc_macro':    roc_auc_score(labels, probs, multi_class='ovr', average='macro'),
        'recall_mel':   recall_score(labels, preds, average=None, zero_division=0)[CLASSES.index('mel')],
        'recall_bcc':   recall_score(labels, preds, average=None, zero_division=0)[CLASSES.index('bcc')],
    }

print(f'Primary weight:   {W_PRIMARY}')
print(f'Secondary weight: {W_SECONDARY}')
print('Selective ensemble defined OK')

Primary weight:   0.6
Secondary weight: 0.4
Selective ensemble defined OK


In [4]:
# ── Extra imports for validation-set inference (needed to fix the weight search) ──
# nb09 originally didn't load models; the weight grid search must run on VALIDATION,
# not on the test set, to avoid selecting hyper-parameters on test (data leakage).
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from config import SPLITS_DIR, MODELS_DIR, BATCH_SIZE, NUM_WORKERS
from dataset import MultimodalSkinLesionDataset
from model import MultimodalModel
from transforms import get_val_transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

/home/maialen/pfg-venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


/home/maialen/pfg-venv/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [5]:
# ── Per-model probabilities on the VALIDATION set (fold 0) ────────────────────
# Same inference as nb10's get_val_ensemble_probs, but returning the two probability
# matrices SEPARATELY (before ensembling) so we can grid-search the weights on val.
def get_val_model_probs(metadata_cols_primary, metadata_cols_secondary,
                        exp_name_primary, exp_name_secondary, fold=0):
    """
    Run inference for each model on fold_{fold}_val.csv.
    Returns: probs_primary (Nval, 7), probs_secondary (Nval, 7), labels (Nval,)
    """
    val_df   = pd.read_csv(os.path.join(SPLITS_DIR, f'fold_{fold}_val.csv'))
    age_mean = pd.read_csv(os.path.join(SPLITS_DIR, f'fold_{fold}_train.csv'))['age'].mean()

    out = {}
    val_labels = None
    for exp_name, metadata_cols in [
        (exp_name_primary,   metadata_cols_primary),
        (exp_name_secondary, metadata_cols_secondary),
    ]:
        dataset = MultimodalSkinLesionDataset(
            val_df, 'none', metadata_cols, get_val_transforms(), age_mean
        )
        loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

        metadata_dim = dataset.get_metadata_dim()
        model_path   = os.path.join(MODELS_DIR, f'{exp_name}_fold{fold}.pth')
        model = MultimodalModel(metadata_dim=metadata_dim, efficientnet_version='b0', pretrained=False)
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=False))
        model = model.to(device).eval()

        probs, labs = [], []
        with torch.no_grad():
            for images, metadata, lbls in tqdm(loader, desc=f'Val {exp_name}', leave=False):
                images, metadata = images.to(device), metadata.to(device)
                p = torch.softmax(model(images, metadata), dim=1).cpu().numpy()
                probs.extend(p); labs.extend(lbls.numpy())

        out[exp_name] = np.array(probs)
        val_labels = np.array(labs)
        model.cpu(); del model; torch.cuda.empty_cache()

    return out[exp_name_primary], out[exp_name_secondary], val_labels


val_probs_loc, val_probs_sex, val_labels = get_val_model_probs(
    metadata_cols_primary   = ['localization'],
    metadata_cols_secondary = ['sex', 'age'],
    exp_name_primary        = 'multimodal_b0_none_localization',
    exp_name_secondary      = 'multimodal_b0_none_sex_age',
    fold=0,
)
print(f'Val samples: {len(val_labels)} | loc {val_probs_loc.shape} | sex {val_probs_sex.shape}')

Val samples: 1696 | loc (1696, 7) | sex (1696, 7)


## Weight Search

Grid search over possible weight combinations to find the optimal weights for the selective weighted ensemble.
Weights are searched on the **standard probabilities** and the best combination is then applied to both standard and TTA probabilities.
Optimization metric: **Recall Melanoma** (primary clinical priority).

In [6]:
# ── Weight grid search on VALIDATION (no test leakage) ────────────────────────
# Same grid as before, but evaluated on val_probs_* / val_labels (fold-0 validation),
# so the 0.7/0.3 choice is justified WITHOUT looking at the test set.
weight_combinations = [
    (0.5, 0.5),
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2),
    (0.9, 0.1),
]

search_results = []
for w1, w2 in weight_combinations:
    final_probs, agree_mask = selective_weighted_ensemble(
        val_probs_loc, val_probs_sex, w1=w1, w2=w2          # <-- VALIDATION, not test
    )
    metrics = compute_metrics(val_labels, final_probs)       # <-- val_labels
    n_agree    = agree_mask.sum()
    n_disagree = (~agree_mask).sum()
    search_results.append({
        'w_primary':   w1,
        'w_secondary': w2,
        'recall_mel':  round(metrics['recall_mel'], 4),
        'recall_bcc':  round(metrics['recall_bcc'], 4),
        'auc_macro':   round(metrics['auc_macro'], 4),
        'recall_macro':round(metrics['recall_macro'], 4),
        'f1_macro':    round(metrics['f1_macro'], 4),
        'n_agree':     n_agree,
        'n_disagree':  n_disagree,
    })

df_search = pd.DataFrame(search_results)
print('Weight search on VALIDATION (fold 0) — no test leakage:')
print(df_search.to_string(index=False))
print(f"\nBest weights by Recall Melanoma (validation):")
best_row = df_search.loc[df_search['recall_mel'].idxmax()]
print(f"  w_primary={best_row['w_primary']}, w_secondary={best_row['w_secondary']} -> Recall mel={best_row['recall_mel']}")

Weight search on VALIDATION (fold 0) — no test leakage:
 w_primary  w_secondary  recall_mel  recall_bcc  auc_macro  recall_macro  f1_macro  n_agree  n_disagree
       0.5          0.5      0.6898      0.8778     0.9650        0.7367    0.7104     1470         226
       0.6          0.4      0.6952      0.8667     0.9649        0.7373    0.7135     1470         226
       0.7          0.3      0.6898      0.8667     0.9648        0.7361    0.7127     1470         226
       0.8          0.2      0.6952      0.8444     0.9646        0.7318    0.7004     1470         226
       0.9          0.1      0.7005      0.8444     0.9643        0.7319    0.6970     1470         226

Best weights by Recall Melanoma (validation):
  w_primary=0.9, w_secondary=0.1 -> Recall mel=0.7005


### Weight Search Analysis — Delta Table

In [10]:
# ── Delta analysis on VALIDATION + multi-criterion selection (no hardcoded winner) ──
# We do NOT hardcode 0.7/0.3 anymore. We report several criteria and let the table
# decide, because on validation the metrics turned out to be almost flat w.r.t. weights.
df_delta = df_search.copy()

df_delta['delta_mel'] = df_delta['recall_mel'].diff().round(4)
df_delta['delta_bcc'] = df_delta['recall_bcc'].diff().round(4)
df_delta['delta_auc'] = df_delta['auc_macro'].diff().round(4)
df_delta['ratio_mel_bcc'] = (df_delta['delta_mel'] / df_delta['delta_bcc'].abs()).round(3)

# Balanced objectives that do NOT reward a degenerate extreme:
df_delta['recall_mel_bcc'] = (df_delta['recall_mel'] + df_delta['recall_bcc']).round(4)

def fmt(x):
    if pd.isna(x):
        return 'N/A'
    return f'+{x:.4f}' if x > 0 else f'{x:.4f}'

df_display = df_delta[['w_primary', 'w_secondary', 'recall_mel', 'recall_bcc',
                       'recall_macro', 'f1_macro', 'recall_mel_bcc',
                       'delta_mel', 'delta_bcc', 'ratio_mel_bcc']].copy()
df_display['delta_mel']     = df_delta['delta_mel'].apply(fmt)
df_display['delta_bcc']     = df_delta['delta_bcc'].apply(fmt)
df_display['ratio_mel_bcc'] = df_delta['ratio_mel_bcc'].apply(lambda x: f'{x:.3f}' if not pd.isna(x) else 'N/A')

print('Weight search — Delta analysis on VALIDATION:')
print('(metrics are nearly flat across weights; we pick by a balanced criterion)\n')
print(df_display.to_string(index=False))

# Spread of each metric across the whole grid: if tiny, weights barely matter.
print('\nMetric spread across the weight grid (max - min):')
for col in ['recall_mel', 'recall_bcc', 'recall_macro', 'f1_macro', 'auc_macro']:
    spread = df_search[col].max() - df_search[col].min()
    print(f'  {col:<13} {spread:.4f}')

# Report the winner under several criteria (no hardcoding):
print('\nBest weights under different criteria (validation):')
for crit in ['recall_mel', 'recall_bcc', 'recall_macro', 'f1_macro', 'recall_mel_bcc']:
    r = df_delta.loc[df_delta[crit].idxmax()]
    print(f'  by {crit:<15} -> w={r["w_primary"]}/{r["w_secondary"]}  ({crit}={r[crit]:.4f})')

Weight search — Delta analysis on VALIDATION:
(metrics are nearly flat across weights; we pick by a balanced criterion)

 w_primary  w_secondary  recall_mel  recall_bcc  recall_macro  f1_macro  recall_mel_bcc delta_mel delta_bcc ratio_mel_bcc
       0.5          0.5      0.6898      0.8778        0.7367    0.7104          1.5676       N/A       N/A           N/A
       0.6          0.4      0.6952      0.8667        0.7373    0.7135          1.5619   +0.0054   -0.0111         0.486
       0.7          0.3      0.6898      0.8667        0.7361    0.7127          1.5565   -0.0054    0.0000          -inf
       0.8          0.2      0.6952      0.8444        0.7318    0.7004          1.5396   +0.0054   -0.0223         0.242
       0.9          0.1      0.7005      0.8444        0.7319    0.6970          1.5449   +0.0053    0.0000           inf

Metric spread across the weight grid (max - min):
  recall_mel    0.0107
  recall_bcc    0.0334
  recall_macro  0.0055
  f1_macro      0.0165
  au

### Weight Decision: 0.6 / 0.4 (selected on validation)

The original 0.7/0.3 choice was made on the **test set**, which is a model-selection
leak. The weight search was therefore re-run on the **fold-0 validation set**. On
validation, the ensemble is almost **insensitive to the weights** — the spread across
the whole grid is tiny for every metric:

| Metric | Spread (max − min) across weights |
|---|---|
| AUC macro | 0.0007 |
| Recall macro | 0.0055 |
| Recall mel | 0.0107 |
| F1 macro | 0.0165 |
| Recall BCC | 0.0334 |

Validation results per weight combination:

| Weights | Recall Mel | Recall BCC | Recall macro | F1 macro |
|---------|-----------|-----------|--------------|----------|
| 0.5 / 0.5 | 0.6898 | 0.8778 | 0.7367 | 0.7104 |
| **0.6 / 0.4** | **0.6952** | 0.8667 | **0.7373** | **0.7135** |
| 0.7 / 0.3 | 0.6898 | 0.8667 | 0.7361 | 0.7127 |
| 0.8 / 0.2 | 0.6952 | 0.8444 | 0.7318 | 0.7004 |
| 0.9 / 0.1 | 0.7005 | 0.8444 | 0.7319 | 0.6970 |

**Selected: w_primary = 0.6 / w_secondary = 0.4.** Because the metrics are essentially
flat, the weights are chosen by a balanced criterion rather than by a single class:
0.6/0.4 maximizes both **recall macro** and **F1 macro** on validation. The extreme
0.9/0.1 wins recall mel only by noise (0.0107 spread) while degrading recall BCC and F1,
so it is rejected. This selection is made **without using the test set**.

In [8]:
# ── Final ensemble weights (selected on VALIDATION, fold 0) ───────────────────
# Validation showed the metrics are almost flat across weights
# (spread: auc 0.0007, recall_macro 0.0055, recall_mel 0.0107, recall_bcc 0.0334).
# Among non-degenerate options, 0.6/0.4 maximizes recall_macro and F1 macro on
# validation, so it is the selected operating point (avoiding the 0.9/0.1 extreme
# that only wins recall_mel by noise while hurting recall_bcc and F1).
W_PRIMARY   = 0.6  # localization (primary)
W_SECONDARY = 0.4  # sex_age (secondary)

print(f'Selected ensemble weights (validation): w_primary={W_PRIMARY}, w_secondary={W_SECONDARY}')

Selected ensemble weights (validation): w_primary=0.6, w_secondary=0.4


## Applying Selective Weighted Ensemble (w=0.7/0.3)

Applying the selective weighted ensemble with the optimal weights to both standard and TTA probabilities.
Results at default threshold 0.5 (experiments 1 and 2). Threshold adjustment is covered in Notebook 10.

In [11]:
# Experiment 1: Standard + threshold 0.5
ensemble_std_probs, agree_mask_std = selective_weighted_ensemble(
    std_probs_loc, std_probs_sex, w1=W_PRIMARY, w2=W_SECONDARY
)
metrics_std = compute_metrics(labels, ensemble_std_probs)

# Experiment 2: TTA + threshold 0.5
ensemble_tta_probs, agree_mask_tta = selective_weighted_ensemble(
    tta_probs_loc, tta_probs_sex, w1=W_PRIMARY, w2=W_SECONDARY
)
metrics_tta = compute_metrics(labels, ensemble_tta_probs)

# Agreement stats
print(f"Standard — Agree: {agree_mask_std.sum()} ({agree_mask_std.mean()*100:.1f}%) | Disagree: {(~agree_mask_std).sum()} ({(~agree_mask_std).mean()*100:.1f}%)")
print(f"TTA      — Agree: {agree_mask_tta.sum()} ({agree_mask_tta.mean()*100:.1f}%) | Disagree: {(~agree_mask_tta).sum()} ({(~agree_mask_tta).mean()*100:.1f}%)")

print(f"\n{'Metric':<15} {'Std ensemble':>14} {'TTA ensemble':>14}")
print(f"{'-'*45}")
for metric in ['auc_macro', 'recall_macro', 'recall_mel', 'recall_bcc', 'f1_macro']:
    print(f"{metric:<15} {metrics_std[metric]:>14.4f} {metrics_tta[metric]:>14.4f}")

Standard — Agree: 1304 (87.1%) | Disagree: 193 (12.9%)
TTA      — Agree: 1312 (87.6%) | Disagree: 185 (12.4%)

Metric            Std ensemble   TTA ensemble
---------------------------------------------
auc_macro               0.9623         0.9668
recall_macro            0.7484         0.7315
recall_mel              0.7679         0.7381
recall_bcc              0.8182         0.7922
f1_macro                0.7230         0.7162


## Saving Ensemble Probabilities

Ensemble probabilities saved to `outputs/metrics/` for use in **Notebook 10 (Threshold Adjustment)**.

| File | Description |
|------|-------------|
| `ensemble_std_probs.npy` | Ensemble probabilities — standard |
| `ensemble_tta_probs.npy` | Ensemble probabilities — TTA |
| `ensemble_labels.npy` | Ground truth labels (test set) |

In [ ]:
np.save(os.path.join(METRICS_DIR, 'ensemble_std_probs.npy'), ensemble_std_probs)
np.save(os.path.join(METRICS_DIR, 'ensemble_tta_probs.npy'), ensemble_tta_probs)
np.save(os.path.join(METRICS_DIR, 'ensemble_labels.npy'), labels)

print("Saved:")
print(f"  ensemble_std_probs.npy -> {METRICS_DIR}")
print(f"  ensemble_tta_probs.npy -> {METRICS_DIR}")
print(f"  ensemble_labels.npy -> {METRICS_DIR}")

Saved:
  ensemble_std_probs.npy -> /home/maialen/skin_lesion_PFG/outputs/metrics
  ensemble_tta_probs.npy -> /home/maialen/skin_lesion_PFG/outputs/metrics
  ensemble_labels.npy -> /home/maialen/skin_lesion_PFG/outputs/metrics


: 